# JSSP Branch & Bound - Large Instances (LA16-LA20)
## Memory-Optimized Configuration

**Instances**: LA16-LA20 (10x10, very challenging)

**Memory Optimizations**:
- Reduced timeout: 1800s per instance (30 minutes)
- Aggressive pruning
- Limited search depth

**Expected Results**:
- Most instances will timeout
- Goal: Find good feasible solutions (not necessarily optimal)
- Gap to BKS: 5-15% expected

In [ ]:
import os
import sys
import time
import logging
from pathlib import Path
from dataclasses import dataclass, field
from typing import Optional, Literal
import pandas as pd
import gc

# Configuration - REDUCED TIMEOUT for large instances
OUTPUT_PATH = "bnb_large_results.csv"
TIMEOUT = 1800  # 30 minutes per instance (reduced from 3600s)

# Setup logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)

print("Setup complete!")
print(f"Output file: {OUTPUT_PATH}")
print(f"Timeout: {TIMEOUT}s per instance (30 minutes)")
print("⚠️ These are LARGE instances (10x10) - expect timeouts!")

In [ ]:
# Embedded instance data - LA16-LA20 only
INSTANCES = {'LA16': '10 10\n1 21 6 71 9 16 8 52 7 26 2 34 0 53 4 21 3 55 5 95\n4 55 2 31 5 98 9 79 0 12 7 66 1 42 8 77 6 77 3 39\n3 34 2 64 8 62 1 19 4 92 9 79 7 43 6 54 0 83 5 37\n1 87 3 69 2 87 7 38 8 24 9 83 6 41 0 93 5 77 4 60\n2 98 0 44 5 25 6 75 7 43 1 49 4 96 9 77 3 17 8 79\n2 35 3 76 5 28 9 10 4 61 6 9 0 95 8 35 1 7 7 95\n3 16 2 59 0 46 1 91 9 43 8 50 6 52 5 59 4 28 7 27\n1 45 0 87 3 41 4 20 6 54 9 43 8 14 5 9 2 39 7 71\n4 33 2 37 8 66 5 33 3 26 7 8 1 28 6 89 9 42 0 78\n8 69 9 81 2 94 4 96 3 27 0 69 7 45 6 78 1 74 5 84', 'LA17': '10 10\n4 18 7 21 9 41 2 45 3 38 8 50 5 84 6 29 1 23 0 82\n8 57 5 16 1 52 7 74 2 38 3 54 6 62 9 37 4 54 0 52\n2 30 4 79 3 68 1 61 8 11 6 89 7 89 0 81 9 81 5 57\n0 91 8 8 3 33 7 55 5 20 2 20 4 32 6 84 1 66 9 24\n9 40 0 7 4 19 8 7 6 83 2 64 5 56 3 54 7 8 1 39\n3 91 2 64 5 40 0 63 7 98 4 74 8 61 1 6 6 42 9 15\n1 80 7 39 8 24 3 75 4 75 5 6 6 44 0 26 2 87 9 22\n1 15 7 43 2 20 0 12 8 26 6 61 3 79 9 22 5 8 4 80\n2 62 3 96 4 22 9 5 0 63 6 33 7 10 8 18 1 36 5 40\n1 96 0 89 5 64 3 95 9 23 7 18 8 15 2 64 6 38 4 8', 'LA18': '10 10\n6 54 0 87 4 48 3 60 7 39 8 35 1 72 5 95 2 66 9 5\n3 20 9 46 6 34 5 55 0 97 8 19 4 59 2 21 7 37 1 46\n4 45 1 24 8 28 0 28 7 83 6 78 5 23 3 25 9 5 2 73\n9 12 1 37 4 38 3 71 8 33 2 12 6 55 0 53 7 87 5 29\n3 83 2 49 6 23 9 27 7 65 0 48 4 90 5 7 1 40 8 17\n1 66 4 25 0 62 2 84 9 13 6 64 7 46 8 59 5 19 3 85\n1 73 3 80 0 41 2 53 9 47 7 57 8 74 4 14 6 67 5 88\n5 64 3 84 6 46 1 78 0 84 7 26 8 28 9 52 2 41 4 63\n1 11 0 64 7 67 4 85 3 10 5 73 9 38 8 95 6 97 2 17\n4 60 8 32 2 95 3 93 1 65 6 85 7 43 9 85 5 46 0 59', 'LA19': '10 10\n2 44 3 5 5 58 4 97 0 9 7 84 8 77 9 96 1 58 6 89\n4 15 7 31 1 87 8 57 0 77 3 85 2 81 5 39 9 73 6 21\n9 82 6 22 4 10 3 70 1 49 0 40 8 34 2 48 7 80 5 71\n1 91 2 17 7 62 5 75 8 47 4 11 3 7 6 72 9 35 0 55\n6 71 1 90 3 75 0 64 2 94 8 15 4 12 7 67 9 20 5 50\n7 70 5 93 8 77 2 29 4 58 6 93 3 68 1 57 9 7 0 52\n6 87 1 63 4 26 5 6 2 82 3 27 7 56 8 48 9 36 0 95\n0 36 5 15 8 41 9 78 3 76 6 84 4 30 7 76 2 36 1 8\n5 88 2 81 3 13 6 82 4 54 7 13 8 29 9 40 1 78 0 75\n9 88 4 54 6 64 7 32 0 52 2 6 8 54 5 82 3 6 1 26', 'LA20': '10 10\n6 9 1 81 4 55 2 40 8 32 3 37 0 6 5 19 9 81 7 40\n7 21 2 70 9 65 4 64 1 46 5 65 8 25 0 77 3 55 6 15\n2 85 5 37 0 40 3 24 1 44 6 83 4 89 8 31 7 84 9 29\n4 80 6 77 7 56 0 8 2 30 5 59 3 38 1 80 9 41 8 97\n0 91 6 40 4 88 1 17 2 71 3 50 9 59 8 80 5 56 7 7\n2 8 6 9 3 58 5 77 1 29 8 96 0 45 9 10 4 54 7 36\n4 70 3 92 1 98 5 87 6 99 7 27 8 86 9 96 0 28 2 73\n1 95 7 92 3 85 4 52 6 81 9 32 8 39 0 59 2 41 5 56\n3 60 8 45 0 88 2 12 1 7 5 22 4 93 9 49 7 69 6 27\n0 21 2 61 3 68 5 26 6 82 9 71 8 44 4 99 7 33 1 84'}

BKS = {'LA16': 945, 'LA17': 784, 'LA18': 848, 'LA19': 842, 'LA20': 902}

print(f"Loaded {len(INSTANCES)} large instances: {list(INSTANCES.keys())}")
print("Instance sizes: All 10x10 (100 operations each)")

In [ ]:
"""Disjunctive Graph for JSSP. Brucker et al. (1994), Carlier & Pinson (1989)."""
from __future__ import annotations
from dataclasses import dataclass, field
from typing import Optional
from collections import deque


@dataclass
class Operation:
    job: int; pos: int; machine: int; duration: int; op_id: int
    def __repr__(self) -> str:
        return f"O(j{self.job},m{self.machine},d={self.duration},id={self.op_id})"

@dataclass
class JSSPInstance:
    name: str; num_jobs: int; num_machines: int
    operations: list[list[Operation]]
    all_ops: list[Operation] = field(default_factory=list)
    machine_ops: dict[int, list[int]] = field(default_factory=dict)
    bks: Optional[int] = None
    def __post_init__(self) -> None:
        self.all_ops = []
        self.machine_ops = {m: [] for m in range(self.num_machines)}
        for jo in self.operations:
            for op in jo:
                self.all_ops.append(op)
                self.machine_ops[op.machine].append(op.op_id)
    @property
    def num_ops(self) -> int: return len(self.all_ops)


class DisjunctiveGraph:
    SOURCE = -1; SINK = -2

    def __init__(self, instance: JSSPInstance) -> None:
        self.instance = instance
        n = instance.num_ops
        self.ops = instance.all_ops
        self.conj_succ = [self.SINK] * n
        self.conj_pred = [self.SOURCE] * n
        for jo in instance.operations:
            for i in range(len(jo) - 1):
                self.conj_succ[jo[i].op_id] = jo[i+1].op_id
                self.conj_pred[jo[i+1].op_id] = jo[i].op_id
        self.disj_succ: list[list[int]] = [[] for _ in range(n)]
        self.disj_pred: list[list[int]] = [[] for _ in range(n)]
        self.heads = [0] * n
        self.tails = [0] * n
        self.fixed_arcs: set[tuple[int,int]] = set()

    def copy(self) -> DisjunctiveGraph:
        g = DisjunctiveGraph.__new__(DisjunctiveGraph)
        g.instance = self.instance; g.ops = self.ops
        g.conj_succ = self.conj_succ; g.conj_pred = self.conj_pred
        g.disj_succ = [l[:] for l in self.disj_succ]
        g.disj_pred = [l[:] for l in self.disj_pred]
        g.heads = self.heads[:]; g.tails = self.tails[:]
        g.fixed_arcs = set(self.fixed_arcs)
        return g

    def fix_arc(self, i: int, j: int) -> None:
        if (i,j) not in self.fixed_arcs:
            self.fixed_arcs.add((i,j))
            self.disj_succ[i].append(j)
            self.disj_pred[j].append(i)

    def is_fixed(self, i: int, j: int) -> bool:
        return (i,j) in self.fixed_arcs

    def compute_heads_and_tails(self) -> bool:
        n = len(self.ops)
        # --- heads (forward topological) ---
        indeg = [0]*n
        for i in range(n):
            if self.conj_pred[i] != self.SOURCE: indeg[i] += 1
            indeg[i] += len(self.disj_pred[i])
        self.heads = [0]*n
        q = deque(i for i in range(n) if indeg[i]==0)
        cnt = 0
        while q:
            i = q.popleft(); cnt += 1
            fi = self.heads[i] + self.ops[i].duration
            cs = self.conj_succ[i]
            if cs != self.SINK:
                self.heads[cs] = max(self.heads[cs], fi)
                indeg[cs] -= 1
                if indeg[cs]==0: q.append(cs)
            for j in self.disj_succ[i]:
                self.heads[j] = max(self.heads[j], fi)
                indeg[j] -= 1
                if indeg[j]==0: q.append(j)
        if cnt != n: return False
        # --- tails (reverse topological) ---
        outdeg = [0]*n
        for i in range(n):
            if self.conj_succ[i] != self.SINK: outdeg[i] += 1
            outdeg[i] += len(self.disj_succ[i])
        self.tails = [0]*n
        q = deque(i for i in range(n) if outdeg[i]==0)
        cnt = 0
        while q:
            i = q.popleft(); cnt += 1
            cp = self.conj_pred[i]
            if cp != self.SOURCE:
                self.tails[cp] = max(self.tails[cp], self.ops[i].duration + self.tails[i])
                outdeg[cp] -= 1
                if outdeg[cp]==0: q.append(cp)
            for j in self.disj_pred[i]:
                self.tails[j] = max(self.tails[j], self.ops[i].duration + self.tails[i])
                outdeg[j] -= 1
                if outdeg[j]==0: q.append(j)
        return cnt == n

    def makespan_lb(self) -> int:
        return max(self.heads[i]+self.ops[i].duration+self.tails[i]
                   for i in range(len(self.ops)))

    def unfixed_on_machine(self, m: int) -> list[tuple[int,int]]:
        ops = self.instance.machine_ops[m]
        pairs = []
        for i in range(len(ops)):
            for j in range(i+1, len(ops)):
                a,b = ops[i],ops[j]
                if not self.is_fixed(a,b) and not self.is_fixed(b,a):
                    pairs.append((a,b))
        return pairs

    def has_unfixed(self) -> bool:
        for m in range(self.instance.num_machines):
            if self.unfixed_on_machine(m):
                return True
        return False

    def most_critical_pair(self) -> Optional[tuple[int,int]]:
        """Carlier & Pinson §3.6: maximize |d_ij - d_ji|, tiebreak min(d_ij,d_ji)."""
        best = None; bv = -1; ba = -1
        for m in range(self.instance.num_machines):
            for a,b in self.unfixed_on_machine(m):
                d_ab = self.heads[a]+self.ops[a].duration+self.ops[b].duration+self.tails[b]
                d_ba = self.heads[b]+self.ops[b].duration+self.ops[a].duration+self.tails[a]
                v = abs(d_ab - d_ba); av = min(d_ab, d_ba)
                if v > bv or (v == bv and av > ba):
                    bv = v; ba = av; best = (a,b)
        return best


def parse_instance(name: str, data: str, bks: Optional[int] = None) -> JSSPInstance:
    lines = data.strip().split('\n')
    hdr = lines[0].strip().split()
    nj, nm = int(hdr[0]), int(hdr[1])
    ops: list[list[Operation]] = []; oid = 0
    for j in range(nj):
        toks = lines[j+1].strip().split()
        row = [Operation(job=j,pos=k,machine=int(toks[2*k]),
                         duration=int(toks[2*k+1]),op_id=oid+k) for k in range(nm)]
        oid += nm; ops.append(row)
    return JSSPInstance(name=name,num_jobs=nj,num_machines=nm,operations=ops,bks=bks)

In [ ]:
"""
Constraint Propagation: Immediate Selection + JPS Lower Bound + Edge-Finding.
Carlier & Pinson (1989), Brucker et al. (1994) §5-6.

Edge-Finding rules implemented:
  NOT-LAST:  if r_min(J∪{i}) + p(J∪{i}) + q_i >= UB → i not last  → q_i >= min_{j∈J}(p_j+q_j)
  NOT-FIRST: if q_min(J∪{i}) + p(J∪{i}) + r_i >= UB → i not first → r_i >= min_{j∈J}(r_j+p_j)
"""
from __future__ import annotations
from algorithms.bnb.graph import DisjunctiveGraph
from itertools import combinations
import heapq


def jackson_preemptive(rs: list[int], ps: list[int], qs: list[int]) -> int:
    """1|r_j,q_j,pmtn|Cmax. Returns max(C_j+q_j). O(n log n)."""
    n = len(rs)
    if n == 0: return 0
    order = sorted(range(n), key=lambda i: rs[i])
    pq: list[tuple[int,int,int]] = []  # (-q, rem_p, idx)
    ji = 0; t = rs[order[0]]; cmax = 0
    while pq or ji < n:
        while ji < n and rs[order[ji]] <= t:
            j = order[ji]; heapq.heappush(pq, (-qs[j], ps[j], j)); ji += 1
        if not pq:
            if ji < n: t = rs[order[ji]]; continue
            else: break
        nq, rp, idx = heapq.heappop(pq)
        nxt = rs[order[ji]] if ji < n else t + rp
        avail = nxt - t
        if avail >= rp:
            t += rp; cmax = max(cmax, t + (-nq))
        else:
            t = nxt; heapq.heappush(pq, (nq, rp - avail, idx))
    return cmax


def jps_lower_bound(graph: DisjunctiveGraph) -> int:
    lb = 0
    for m in range(graph.instance.num_machines):
        ops = graph.instance.machine_ops[m]
        if len(ops) < 1: continue
        rs = [graph.heads[i] for i in ops]
        ps = [graph.ops[i].duration for i in ops]
        qs = [graph.tails[i] for i in ops]
        lb = max(lb, jackson_preemptive(rs, ps, qs))
    return lb


def immediate_selection(graph: DisjunctiveGraph, ub: int) -> bool:
    """
    Brucker (1994) §5, Lemma 5.2:
    For ops i,j on same machine, if r_i + p_i + p_j + q_j >= ub
    then i before j is impossible => fix j -> i.

    Returns False if cycle detected (infeasible).
    """
    changed = True
    while changed:
        changed = False
        for m in range(graph.instance.num_machines):
            ops = graph.instance.machine_ops[m]
            for ai in range(len(ops)):
                a = ops[ai]
                ra = graph.heads[a]; pa = graph.ops[a].duration; qa = graph.tails[a]
                for bi in range(ai+1, len(ops)):
                    b = ops[bi]
                    if graph.is_fixed(a,b) or graph.is_fixed(b,a):
                        continue
                    rb = graph.heads[b]; pb = graph.ops[b].duration; qb = graph.tails[b]
                    # Can a be before b? need ra+pa+pb+qb < ub
                    a_before_b_lb = ra + pa + pb + qb
                    b_before_a_lb = rb + pb + pa + qa
                    if a_before_b_lb >= ub and b_before_a_lb >= ub:
                        return False  # Both impossible => infeasible
                    if a_before_b_lb >= ub:
                        # a before b impossible => b must be before a
                        graph.fix_arc(b, a)
                        changed = True
                    elif b_before_a_lb >= ub:
                        # b before a impossible => a must be before b
                        graph.fix_arc(a, b)
                        changed = True
        if changed:
            if not graph.compute_heads_and_tails():
                return False
    return True


def edge_finding(graph: DisjunctiveGraph, ub: int) -> bool:
    """
    NOT-LAST and NOT-FIRST edge-finding rules (Carlier & Pinson 1989).
    Updates graph.heads and graph.tails in-place.
    Returns False if infeasibility is detected.

    For machines with <= 9 ops: enumerate all subsets J of Omega_m minus {i}.
    For larger machines: check singletons, pairs, triples, and full set only.
    """
    for m in range(graph.instance.num_machines):
        ops = graph.instance.machine_ops[m]
        n = len(ops)
        if n < 2:
            continue

        p = [graph.ops[o].duration for o in ops]
        p_map = {ops[k]: p[k] for k in range(n)}

        # Decide subset sizes: full enum for small, limited for large
        max_others = n - 1
        if max_others <= 8:
            sizes = list(range(1, max_others + 1))
        else:
            sizes = list(range(1, 4)) + [max_others]

        for i in ops:
            others = [o for o in ops if o != i]
            pi = p_map[i]

            for sz in sizes:
                if sz > len(others):
                    break
                for J in combinations(others, sz):
                    # Inline aggregation with plain loops — avoids repeated
                    # generator-object allocation inside the hot inner loop.
                    p_J = 0
                    r_min = graph.heads[i]
                    q_min = graph.tails[i]
                    for j in J:
                        p_J += p_map[j]
                        hj = graph.heads[j]
                        if hj < r_min:
                            r_min = hj
                        tj = graph.tails[j]
                        if tj < q_min:
                            q_min = tj
                    p_total = p_J + pi

                    # NOT-LAST: r_min(J∪{i}) + p(J∪{i}) + q_i >= ub
                    if r_min + p_total + graph.tails[i] >= ub:
                        min_pq = graph.tails[i]  # sentinel — will be beaten by any j
                        for j in J:
                            pq = p_map[j] + graph.tails[j]
                            if pq < min_pq:
                                min_pq = pq
                        if min_pq > graph.tails[i]:
                            graph.tails[i] = min_pq

                    # NOT-FIRST: q_min(J∪{i}) + p(J∪{i}) + r_i >= ub
                    if q_min + p_total + graph.heads[i] >= ub:
                        min_rp = graph.heads[i]  # sentinel
                        for j in J:
                            rp = graph.heads[j] + p_map[j]
                            if rp < min_rp:
                                min_rp = rp
                        if min_rp > graph.heads[i]:
                            graph.heads[i] = min_rp

                    if graph.heads[i] + pi + graph.tails[i] >= ub:
                        return False

    return True


def propagate(graph: DisjunctiveGraph, ub: int) -> bool:
    """Propagate constraints. Returns False if infeasible or LB >= ub."""
    if not graph.compute_heads_and_tails():
        return False
    if graph.makespan_lb() >= ub:
        return False
    if not immediate_selection(graph, ub):
        return False
    if graph.makespan_lb() >= ub:
        return False
    if not edge_finding(graph, ub):
        return False
    if graph.makespan_lb() >= ub:
        return False
    return True


def lower_bound(graph: DisjunctiveGraph) -> int:
    """Max of critical-path LB and JPS LB."""
    return max(graph.makespan_lb(), jps_lower_bound(graph))

In [ ]:
"""
Branch and Bound solver for JSSP.
Carlier & Pinson (1989), Brucker et al. (1994).

KEY: Iterative deepening - each B&B pass tries to find solution < current UB.
If no improvement found, optimality is proven.
"""
from __future__ import annotations
import heapq, random, time, logging
from collections import deque
from dataclasses import dataclass, field
from typing import Optional
# Imports removed - all code is embedded in notebook
# Imports removed - all code is embedded in notebook

logger = logging.getLogger("jssp_solver")

@dataclass
class Schedule:
    start_times: list[int]; makespan: int

@dataclass
class SolverResult:
    instance_name: str; makespan: int; schedule: Optional[Schedule]
    computation_time: float; nodes_explored: int; optimal_proven: bool
    bks: Optional[int]; gap_vs_bks: float
    def to_dict(self) -> dict:
        sd = {str(i):t for i,t in enumerate(self.schedule.start_times)} if self.schedule else None
        return {"instance":self.instance_name,"makespan":self.makespan,
                "schedule":sd,"computation_time":round(self.computation_time,3),
                "nodes_explored":self.nodes_explored,"optimal_proven":self.optimal_proven,
                "bks":self.bks,"gap_vs_bks_pct":round(self.gap_vs_bks,2)}


# ---------- Giffler-Thompson (Pinedo 2016 Algorithm 7.1.3) ----------

def giffler_thompson(instance: JSSPInstance, rule: str = "MWKR") -> Schedule:
    n = instance.num_ops
    st = [0]*n
    machine_time = [0]*instance.num_machines
    job_time = [0]*instance.num_jobs
    nxt = [0]*instance.num_jobs  # next operation index per job

    for _ in range(n):
        # Omega: schedulable ops (next in each job that isn't done)
        omega = []
        for j in range(instance.num_jobs):
            if nxt[j] < instance.num_machines:
                omega.append(instance.operations[j][nxt[j]].op_id)
        if not omega:
            break

        # For each op in omega, compute earliest start and completion
        earliest_completion = {}
        for oid in omega:
            op = instance.all_ops[oid]
            es = max(machine_time[op.machine], job_time[op.job])
            earliest_completion[oid] = es + op.duration

        # t* = minimum earliest completion; i* = its machine
        t_star = min(earliest_completion.values())
        i_star = -1
        for oid in omega:
            if earliest_completion[oid] == t_star:
                i_star = instance.all_ops[oid].machine
                break

        # Omega' = ops on machine i* whose earliest start < t*
        omega_prime = []
        for oid in omega:
            op = instance.all_ops[oid]
            if op.machine == i_star:
                es = max(machine_time[op.machine], job_time[op.job])
                if es < t_star:
                    omega_prime.append(oid)

        # Fallback for zero-duration ops: es == t_star (not < t_star), omega_prime can be empty
        if not omega_prime:
            for oid in omega:
                if instance.all_ops[oid].machine == i_star:
                    omega_prime.append(oid)
                    break

        # Pick from omega' based on dispatching rule
        if rule == "SPT":
            chosen = min(omega_prime, key=lambda oid: instance.all_ops[oid].duration)
        elif rule == "LPT":
            chosen = max(omega_prime, key=lambda oid: instance.all_ops[oid].duration)
        elif rule == "FCFS":
            chosen = min(omega_prime, key=lambda oid: instance.all_ops[oid].job)
        else:  # MWKR (default)
            def remaining_work(oid: int) -> int:
                op = instance.all_ops[oid]
                return sum(instance.operations[op.job][k].duration
                          for k in range(op.pos, instance.num_machines))
            chosen = max(omega_prime, key=remaining_work)

        op = instance.all_ops[chosen]
        es = max(machine_time[op.machine], job_time[op.job])
        st[chosen] = es
        machine_time[op.machine] = es + op.duration
        job_time[op.job] = es + op.duration
        nxt[op.job] += 1

    ms = max(st[i] + instance.all_ops[i].duration for i in range(n))
    return Schedule(start_times=st, makespan=ms)


def _giffler_thompson_random(instance: JSSPInstance) -> Schedule:
    """
    One randomized Giffler-Thompson pass: within Omega', choose randomly
    instead of by a fixed dispatching rule.  Running many iterations and
    keeping the best gives a much tighter initial UB than 4 fixed rules.
    """
    n = instance.num_ops
    st = [0] * n
    machine_time = [0] * instance.num_machines
    job_time     = [0] * instance.num_jobs
    nxt          = [0] * instance.num_jobs

    for _ in range(n):
        omega = [instance.operations[j][nxt[j]].op_id
                 for j in range(instance.num_jobs) if nxt[j] < instance.num_machines]
        if not omega:
            break

        ec = {}
        for oid in omega:
            op = instance.all_ops[oid]
            ec[oid] = max(machine_time[op.machine], job_time[op.job]) + op.duration

        t_star = min(ec.values())
        i_star = instance.all_ops[next(oid for oid in omega if ec[oid] == t_star)].machine

        omega_prime = [oid for oid in omega
                       if instance.all_ops[oid].machine == i_star
                       and max(machine_time[i_star], job_time[instance.all_ops[oid].job]) < t_star]
        if not omega_prime:
            omega_prime = [oid for oid in omega if instance.all_ops[oid].machine == i_star]

        chosen = random.choice(omega_prime)
        op = instance.all_ops[chosen]
        es = max(machine_time[op.machine], job_time[op.job])
        st[chosen] = es
        machine_time[op.machine] = es + op.duration
        job_time[op.job]         = es + op.duration
        nxt[op.job] += 1

    ms = max(st[i] + instance.all_ops[i].duration for i in range(n))
    return Schedule(start_times=st, makespan=ms)


def randomized_gt_ub(instance: JSSPInstance,
                     time_limit: float = 1.0,
                     min_iters: int = 200) -> Schedule:
    """
    Run randomized GT for up to *time_limit* seconds (at least *min_iters*
    iterations).  Returns the best Schedule found.
    """
    best: Optional[Schedule] = None
    deadline = time.perf_counter() + time_limit
    i = 0
    while i < min_iters or time.perf_counter() < deadline:
        s = _giffler_thompson_random(instance)
        if best is None or s.makespan < best.makespan:
            best = s
        i += 1
    return best  # type: ignore[return-value]


# ---------- Schedule Validation ----------

def verify_schedule(instance: JSSPInstance, start_times: list[int]) -> bool:
    """Return True if the schedule satisfies all JSSP constraints."""
    n = instance.num_ops
    if len(start_times) != n:
        return False
    for job_ops in instance.operations:
        for k in range(len(job_ops) - 1):
            a, b = job_ops[k], job_ops[k + 1]
            if start_times[b.op_id] < start_times[a.op_id] + a.duration:
                return False
    for m in range(instance.num_machines):
        ops_m = sorted(instance.machine_ops[m], key=lambda oid: start_times[oid])
        for k in range(len(ops_m) - 1):
            a = instance.all_ops[ops_m[k]]; b = instance.all_ops[ops_m[k + 1]]
            if start_times[b.op_id] < start_times[a.op_id] + a.duration:
                return False
    return True


# ---------- N1 Tabu Search (Nowicki & Smutnicki 1996) ----------

def _extract_machine_seqs(instance: JSSPInstance,
                           sched: Schedule) -> list[list[int]]:
    """For each machine, return op_ids sorted by start time in *sched*."""
    seqs: list[list[int]] = [[] for _ in range(instance.num_machines)]
    for op in instance.all_ops:
        seqs[op.machine].append(op.op_id)
    for m in range(instance.num_machines):
        seqs[m].sort(key=lambda oid: sched.start_times[oid])
    return seqs


def _eval_seqs(instance: JSSPInstance,
               machine_seqs: list[list[int]]) -> tuple[int, list[int]]:
    """
    Compute (makespan, start_times) for the given machine orderings.
    Runs Kahn's topological sort on the DAG of job + machine precedences.
    Returns (-1, []) on cycle (infeasible).
    """
    n = instance.num_ops
    indeg = [0] * n
    succ: list[list[int]] = [[] for _ in range(n)]
    dur = [op.duration for op in instance.all_ops]

    for job_ops in instance.operations:
        for k in range(len(job_ops) - 1):
            a = job_ops[k].op_id; b = job_ops[k + 1].op_id
            succ[a].append(b); indeg[b] += 1
    for seq in machine_seqs:
        for k in range(len(seq) - 1):
            a = seq[k]; b = seq[k + 1]
            succ[a].append(b); indeg[b] += 1

    heads = [0] * n
    q: deque[int] = deque(i for i in range(n) if indeg[i] == 0)
    cnt = 0
    while q:
        i = q.popleft(); cnt += 1
        fi = heads[i] + dur[i]
        for j in succ[i]:
            if fi > heads[j]: heads[j] = fi
            indeg[j] -= 1
            if indeg[j] == 0: q.append(j)

    if cnt != n:
        return -1, []
    return max(heads[i] + dur[i] for i in range(n)), heads


def _compute_tails(instance: JSSPInstance,
                   machine_seqs: list[list[int]],
                   dur: list[int]) -> list[int]:
    """
    Compute tail[i] = longest path from end of op i to makespan, via reverse
    topological sort.

    For each original arc  a → b:
      - reverse arc is  b → a  (stored in r_adj[b])
      - a's in-degree in the reverse graph = a's out-degree in the original
        (tracked as indeg_rev[a])
    Start the queue from sinks of the original graph (indeg_rev == 0 ↔ no
    successors in the original), then propagate tail values backward.
    """
    n = instance.num_ops
    r_adj: list[list[int]] = [[] for _ in range(n)]
    indeg_rev = [0] * n            # = out-degree in original graph
    for job_ops in instance.operations:
        for k in range(len(job_ops) - 1):
            a = job_ops[k].op_id; b = job_ops[k + 1].op_id
            r_adj[b].append(a)     # reverse arc: b → a
            indeg_rev[a] += 1      # a has one more outgoing in original
    for seq in machine_seqs:
        for k in range(len(seq) - 1):
            a = seq[k]; b = seq[k + 1]
            r_adj[b].append(a)
            indeg_rev[a] += 1
    tails = [0] * n
    q: deque[int] = deque(i for i in range(n) if indeg_rev[i] == 0)
    while q:
        b = q.popleft()
        wb = dur[b] + tails[b]
        for a in r_adj[b]:         # a is a predecessor of b in original
            if wb > tails[a]: tails[a] = wb
            indeg_rev[a] -= 1
            if indeg_rev[a] == 0: q.append(a)
    return tails


def tabu_search_n1(instance: JSSPInstance,
                   initial: Schedule,
                   time_limit: float,
                   tenure: int = 7) -> Schedule:
    """
    N1 Tabu Search for JSSP.

    N1 neighborhood (Nowicki & Smutnicki 1996):
      Swap two *adjacent* operations on the same machine within any
      critical-path block (maximal same-machine run on the critical path).

    Parameters
    ----------
    instance   : JSSP instance
    initial    : starting schedule (from GT or randomized GT)
    time_limit : wall-clock budget in seconds
    tenure     : tabu tenure — a reversed swap (op_b, op_a) is forbidden
                 for this many iterations after applying (op_a, op_b)

    Returns
    -------
    Best Schedule found within the budget.
    """
    deadline = time.perf_counter() + time_limit
    dur = [op.duration for op in instance.all_ops]

    machine_seqs = _extract_machine_seqs(instance, initial)
    cur_ms, cur_heads = _eval_seqs(instance, machine_seqs)
    if cur_ms < 0:
        return initial  # initial was somehow infeasible

    best_ms   = cur_ms
    best_sched = Schedule(start_times=cur_heads[:], makespan=cur_ms)
    best_seqs  = [seq[:] for seq in machine_seqs]

    # tabu[(op_a, op_b)] = expiry iteration — the swap a→b is forbidden until then
    tabu: dict[tuple[int, int], int] = {}
    iteration  = 0
    no_improve = 0
    restart_thresh = max(60, instance.num_ops * 3)

    while time.perf_counter() < deadline:
        iteration += 1

        # ── Identify critical-path blocks ─────────────────────────────
        tails = _compute_tails(instance, machine_seqs, dur)
        is_crit = [cur_heads[i] + dur[i] + tails[i] == cur_ms
                   for i in range(instance.num_ops)]

        # ── Enumerate N1 moves: adjacent critical pairs per machine ───
        best_move_ms   = 10 ** 9
        best_move_info: tuple | None = None  # (m, k, op_a, op_b, new_heads)

        for m, seq in enumerate(machine_seqs):
            for k in range(len(seq) - 1):
                op_a = seq[k]; op_b = seq[k + 1]
                if not (is_crit[op_a] and is_crit[op_b]):
                    continue            # not a critical-block swap

                key = (op_a, op_b)
                is_tabu = tabu.get(key, 0) > iteration

                # Evaluate swap in-place, undo immediately
                seq[k], seq[k + 1] = op_b, op_a
                new_ms, new_heads = _eval_seqs(instance, machine_seqs)
                seq[k], seq[k + 1] = op_a, op_b

                if new_ms < 0:
                    continue

                # Aspiration: override tabu if this beats the global best
                if is_tabu and new_ms >= best_ms:
                    continue

                if new_ms < best_move_ms:
                    best_move_ms   = new_ms
                    best_move_info = (m, k, op_a, op_b, new_heads)

        if best_move_info is None:
            # No admissible critical move at all — force a restart
            no_improve = restart_thresh  # trigger restart immediately
        else:
            # ── Apply chosen move ─────────────────────────────────────
            m, k, op_a, op_b, new_heads = best_move_info
            machine_seqs[m][k], machine_seqs[m][k + 1] = op_b, op_a
            tabu[(op_b, op_a)] = iteration + tenure  # reverse swap is tabu

            cur_ms    = best_move_ms
            cur_heads = new_heads

            if cur_ms < best_ms:
                best_ms    = cur_ms
                best_sched = Schedule(start_times=cur_heads[:], makespan=cur_ms)
                best_seqs  = [seq[:] for seq in machine_seqs]
                no_improve = 0
            else:
                no_improve += 1

        # ── Restart with random perturbation when stuck ───────────────
        if no_improve >= restart_thresh:
            if time.perf_counter() >= deadline:
                break
            # Start from best, apply a few random adjacent swaps to escape
            machine_seqs = [seq[:] for seq in best_seqs]
            n_perturb = max(3, instance.num_ops // 8)
            for _ in range(n_perturb):
                m2 = random.randrange(instance.num_machines)
                s2 = machine_seqs[m2]
                if len(s2) >= 2:
                    k2 = random.randrange(len(s2) - 1)
                    s2[k2], s2[k2 + 1] = s2[k2 + 1], s2[k2]
            new_ms2, new_heads2 = _eval_seqs(instance, machine_seqs)
            if new_ms2 > 0:
                cur_ms = new_ms2; cur_heads = new_heads2
            else:
                machine_seqs = [seq[:] for seq in best_seqs]
                cur_ms = best_ms; cur_heads = best_sched.start_times[:]
            tabu.clear()
            no_improve = 0

        # Prune stale tabu entries to keep dict small
        if iteration % 200 == 0:
            tabu = {kk: v for kk, v in tabu.items() if v > iteration}

    if not verify_schedule(instance, best_sched.start_times):
        logger.error("Tabu Search produced an infeasible schedule — returning initial")
        return initial
    return best_sched


def _schedule_from_graph_rule(graph: DisjunctiveGraph, rule: str) -> Optional[Schedule]:
    """Build feasible schedule respecting all fixed arcs with a given dispatch rule."""
    inst = graph.instance; n = inst.num_ops
    st = [0]*n; comp = [0]*n; mt = [0]*inst.num_machines

    pc = [0]*n
    suc: list[list[int]] = [[] for _ in range(n)]
    for i in range(n):
        cp = graph.conj_pred[i]
        if cp != DisjunctiveGraph.SOURCE:
            pc[i] += 1; suc[cp].append(i)
        for dp in graph.disj_pred[i]:
            pc[i] += 1; suc[dp].append(i)

    ready = [i for i in range(n) if pc[i]==0]
    for _ in range(n):
        if not ready: return None
        es = {}
        for i in ready:
            op = inst.all_ops[i]
            t = mt[op.machine]
            cp = graph.conj_pred[i]
            if cp != DisjunctiveGraph.SOURCE: t = max(t, comp[cp])
            for dp in graph.disj_pred[i]: t = max(t, comp[dp])
            es[i] = t

        mec = min(es[i]+inst.all_ops[i].duration for i in ready)
        im = -1
        for i in ready:
            if es[i]+inst.all_ops[i].duration == mec:
                im = inst.all_ops[i].machine; break
        conf = [i for i in ready if inst.all_ops[i].machine==im and es[i]<mec]
        if not conf: conf = [min(ready, key=lambda i: es[i])]

        if rule == "SPT":
            chosen = min(conf, key=lambda oid: inst.all_ops[oid].duration)
        elif rule == "LPT":
            chosen = max(conf, key=lambda oid: inst.all_ops[oid].duration)
        elif rule == "FCFS":
            chosen = min(conf, key=lambda oid: inst.all_ops[oid].job)
        else:  # MWKR
            def rw(oid: int) -> int:
                op = inst.all_ops[oid]
                return sum(inst.operations[op.job][k].duration
                          for k in range(op.pos, inst.num_machines))
            chosen = max(conf, key=rw)

        op = inst.all_ops[chosen]
        st[chosen] = es[chosen]
        comp[chosen] = es[chosen] + op.duration
        mt[op.machine] = comp[chosen]
        ready.remove(chosen)
        for s in suc[chosen]:
            pc[s] -= 1
            if pc[s]==0: ready.append(s)

    return Schedule(start_times=st, makespan=max(comp))


def schedule_from_graph(graph: DisjunctiveGraph) -> Optional[Schedule]:
    """Try multiple dispatch rules, return the schedule with best makespan."""
    best: Optional[Schedule] = None
    for rule in ["MWKR", "SPT", "LPT", "FCFS"]:
        s = _schedule_from_graph_rule(graph, rule)
        if s is not None and (best is None or s.makespan < best.makespan):
            best = s
    return best


def critical_path_and_blocks(graph: DisjunctiveGraph,
                              sched: Schedule) -> list[list[int]]:
    """
    Extract blocks from the critical path of *sched* using actual start times.
    A block = maximal run of consecutive same-machine ops on the critical path,
    size >= 2.  Blocks are returned in critical-path order (first block first).

    For each op i we find the critical predecessor j:
      st[j] + p[j] == st[i]  AND  j is either the conjunctive pred or a
      machine predecessor (another op on the same machine in *sched*).
    """
    n = len(graph.ops)
    st = sched.start_times
    ms = sched.makespan
    p = [graph.ops[i].duration for i in range(n)]

    # For each machine, build a map: finish_time -> op_id
    # so we can quickly find which op finishes at a given time on each machine.
    machine_finish: list[dict[int, int]] = [
        {} for _ in range(graph.instance.num_machines)
    ]
    for i in range(n):
        machine_finish[graph.ops[i].machine][st[i] + p[i]] = i

    # Build critical-predecessor map.
    # Prefer machine predecessor so blocks (same-machine runs) form correctly.
    cpred = [-1] * n
    for i in range(n):
        ti = st[i]
        # Machine predecessor takes priority (needed for block structure)
        m = graph.ops[i].machine
        j = machine_finish[m].get(ti, -1)
        if j != -1 and j != i:
            cpred[i] = j
            continue
        # Fallback: conjunctive predecessor
        cp = graph.conj_pred[i]
        if cp != DisjunctiveGraph.SOURCE and st[cp] + p[cp] == ti:
            cpred[i] = cp

    # Find op(s) completing at makespan; trace path backward
    best_path: list[int] = []
    for e in range(n):
        if st[e] + p[e] != ms:
            continue
        path: list[int] = []
        c = e
        while c != -1:
            path.append(c)
            c = cpred[c]
        path.reverse()
        if len(path) > len(best_path):
            best_path = path

    # Decompose best_path into blocks (maximal same-machine runs, size >= 2)
    blocks: list[list[int]] = []
    if not best_path:
        return blocks
    cur_block = [best_path[0]]
    cur_m = graph.ops[best_path[0]].machine
    for idx in range(1, len(best_path)):
        m = graph.ops[best_path[idx]].machine
        if m == cur_m:
            cur_block.append(best_path[idx])
        else:
            if len(cur_block) >= 2:
                blocks.append(cur_block)
            cur_block = [best_path[idx]]
            cur_m = m
    if len(cur_block) >= 2:
        blocks.append(cur_block)
    return blocks


# ---------- Branch and Bound ----------

@dataclass(order=False)
class BBNode:
    lb: int
    seq: int          # insertion counter for stable heap ordering
    depth: int
    graph: DisjunctiveGraph = field(compare=False)

    def __lt__(self, other: "BBNode") -> bool:
        return (self.lb, self.seq) < (other.lb, other.seq)


class BranchAndBoundSolver:
    """
    Iterative B&B (Carlier & Pinson 1989 §3.7):
    Repeatedly search for solutions with makespan < UB.
    When no improvement possible, UB is optimal.
    """
    def __init__(self, instance: JSSPInstance, timeout: float=3600.0,
                 log_interval: int=1000):
        self.inst = instance; self.timeout = timeout
        self.log_interval = log_interval
        self.best: Optional[Schedule] = None
        self.ub = 10**9; self.nodes = 0; self.t0 = 0.0
        self.optimal = False
        self._seq = 0   # tie-break counter for heap ordering

    def _elapsed(self) -> float: return time.time()-self.t0
    def _timeout(self) -> bool: return self._elapsed()>=self.timeout

    def solve(self) -> SolverResult:
        self.t0 = time.time()
        logger.info(f"Solving {self.inst.name}: {self.inst.num_jobs}x{self.inst.num_machines}")

        # Phase 1 — deterministic GT (4 rules, cheap)
        best_s0: Optional[Schedule] = None
        for rule in ["MWKR", "SPT", "LPT", "FCFS"]:
            s = giffler_thompson(self.inst, rule=rule)
            if best_s0 is None or s.makespan < best_s0.makespan:
                best_s0 = s
        self.ub = best_s0.makespan; self.best = best_s0
        logger.info(f"Initial UB (4 GT rules): {self.ub}")

        # Phase 2 — randomized GT: tighten UB before B&B starts.
        # Budget: 5 % of total timeout, min 0.5 s, max 10 s.
        rgt_budget = min(10.0, max(0.5, self.timeout * 0.05))
        rgt = randomized_gt_ub(self.inst, time_limit=rgt_budget)
        if rgt.makespan < self.ub:
            self.ub = rgt.makespan; self.best = rgt
        logger.info(f"UB after randomized GT ({rgt_budget:.1f}s budget): {self.ub}")

        # Phase 3 — N1 Tabu Search: push UB toward optimal before B&B.
        # Budget: 10 % of total timeout, min 1 s, max 60 s.
        if not self._timeout():
            ts_budget = min(60.0, max(1.0, self.timeout * 0.10))
            ts = tabu_search_n1(self.inst, self.best, time_limit=ts_budget)
            if ts.makespan < self.ub:
                self.ub = ts.makespan; self.best = ts
            logger.info(f"UB after Tabu Search ({ts_budget:.1f}s budget): {self.ub}")

        # Root LB
        g0 = DisjunctiveGraph(self.inst)
        g0.compute_heads_and_tails()
        lb0 = lower_bound(g0)
        logger.info(f"Root LB: {lb0}")

        if lb0 >= self.ub:
            self.optimal = True
            return self._result()

        # Iterative: keep trying to improve
        while not self._timeout():
            logger.info(f"B&B pass: UB={self.ub}, time={self._elapsed():.1f}s")
            improved, exhausted = self._search()
            if not improved:
                # Only declare optimal when the heap was fully drained.
                # A timed-out pass that found no improvement is NOT a proof.
                if exhausted:
                    self.optimal = True
                    logger.info(f"Optimal proven: {self.ub}")
                break
            logger.info(f"Improved to {self.ub}, continuing...")

        return self._result()

    def _make_node(self, graph: DisjunctiveGraph, lb: int, depth: int) -> BBNode:
        self._seq += 1
        return BBNode(lb=lb, seq=self._seq, depth=depth, graph=graph)

    def _search(self) -> tuple[bool, bool]:
        """One B&B pass using best-first search (min-heap by LB).

        Returns
        -------
        (improved, exhausted)
          improved  – True if UB was lowered during this pass.
          exhausted – True if the heap was fully drained (search space proven);
                      False if the pass was cut short by timeout.

        Distinguishing these two cases is critical: a timed-out pass that
        found no improvement must NOT be treated as a proof of optimality.
        """
        ub_at_start = self.ub
        root = DisjunctiveGraph(self.inst)
        if not root.compute_heads_and_tails():
            return False, True
        lb = lower_bound(root)
        if lb >= self.ub:
            return False, True

        heap: list[BBNode] = [self._make_node(root, lb, 0)]
        heapq.heapify(heap)

        while heap and not self._timeout():
            node = heapq.heappop(heap)
            self.nodes += 1

            if self.nodes % self.log_interval == 0:
                logger.info(f"  N={self.nodes} UB={self.ub} LB={node.lb} "
                           f"d={node.depth} heap={len(heap)} t={self._elapsed():.1f}s")

            if node.lb >= self.ub:
                continue

            g = node.graph
            if not propagate(g, self.ub):
                continue
            lb = lower_bound(g)
            if lb >= self.ub:
                continue

            # Heuristic solution
            h = schedule_from_graph(g)
            if h and h.makespan < self.ub:
                self.ub = h.makespan; self.best = h
                logger.info(f"  => UB={self.ub} at node {self.nodes} d={node.depth}")

            # If all disjunctions fixed, no further branching needed
            if not g.has_unfixed():
                continue

            # --- Branching ---
            children = self._branch(g, node.depth, h)
            for child in children:
                heapq.heappush(heap, child)

        exhausted = (len(heap) == 0)   # timed-out passes leave nodes behind
        return (self.ub < ub_at_start), exhausted

    def _branch(self, g: DisjunctiveGraph, depth: int,
                h: Optional[Schedule] = None) -> list[BBNode]:
        """
        Try block-based branching (Brucker 1994 §3).
        Fallback: binary disjunction branching (Carlier & Pinson 1989 §3.6).
        h: heuristic schedule already computed (avoids re-computing).
        """
        children: list[BBNode] = []

        # Block-based: reuse existing schedule if provided
        if h is None:
            h = schedule_from_graph(g)
        if h:
            blocks = critical_path_and_blocks(g, h)
            if blocks:
                children = self._branch_blocks(g, blocks, depth)

        # Fallback to disjunction branching
        if not children:
            children = self._branch_pair(g, depth)

        return children

    # ------------------------------------------------------------------
    # Brucker et al. [BJS92/BJS94] specific lower bounds
    # (handbook §10.2.3, p.362). Cheap pre-pruning before child copy.
    # ------------------------------------------------------------------

    @staticmethod
    def _brucker_before_lb(cand: int, others: list[int],
                           g: DisjunctiveGraph) -> int:
        """
        LB when 'cand' moves to the VERY BEGINNING of its block.
        ri + pi + max( max_j(pj+qj),  Σpj + min_j(qj) )   j ∈ others
        """
        if not others:
            return 0
        ri = g.heads[cand]; pi = g.ops[cand].duration
        sum_p  = sum(g.ops[j].duration for j in others)
        max_pq = max(g.ops[j].duration + g.tails[j] for j in others)
        min_q  = min(g.tails[j] for j in others)
        return ri + pi + max(max_pq, sum_p + min_q)

    @staticmethod
    def _brucker_after_lb(cand: int, others: list[int],
                          g: DisjunctiveGraph) -> int:
        """
        LB when 'cand' moves to the VERY END of its block.
        max( max_j(rj+pj),  Σpj + min_j(rj) ) + pi + qi   j ∈ others
        """
        if not others:
            return 0
        qi = g.tails[cand]; pi = g.ops[cand].duration
        sum_p  = sum(g.ops[j].duration for j in others)
        max_rp = max(g.heads[j] + g.ops[j].duration for j in others)
        min_r  = min(g.heads[j] for j in others)
        return max(max_rp, sum_p + min_r) + pi + qi

    def _branch_blocks(self, g: DisjunctiveGraph, blocks: list[list[int]],
                       depth: int) -> list[BBNode]:
        """
        Brucker et al. [BJS92/BJS94] (handbook §10.2.3, p.362):

        For each block B on the critical path generate 2*(|B|-1) children:
          Before-branch (cand → very beginning of B):
              Fix ALL arcs  cand → o  for every o ∈ B, o ≠ cand.
          After-branch  (cand → very end of B):
              Fix ALL arcs  o → cand  for every o ∈ B, o ≠ cand.

        Fixing all intra-block arcs keeps sub-problems non-intersecting
        (no solution explored twice within the same block).

        A tight Brucker-specific lower bound pre-prunes children before
        the expensive graph-copy + topological-sort step.
        """
        children: list[BBNode] = []

        for block in blocks:
            if len(block) < 2:
                continue

            # ── Before-candidates: cand → very BEGINNING of block ────────
            for cand in block[1:]:          # all except current first
                if self._timeout():
                    return children
                others = [o for o in block if o != cand]

                # Cheap Brucker LB pre-prune (no copy required)
                blb = self._brucker_before_lb(cand, others, g)
                if blb >= self.ub:
                    continue

                # Skip if any arc already forces cand to come AFTER an 'other'
                if any(g.is_fixed(o, cand) for o in others):
                    continue

                child = g.copy()
                for o in others:
                    child.fix_arc(cand, o)       # cand → every other in block
                if not child.compute_heads_and_tails():
                    continue
                lb = max(blb, lower_bound(child))
                if lb < self.ub:
                    children.append(self._make_node(child, lb, depth+1))

            # ── After-candidates: cand → very END of block ───────────────
            for cand in block[:-1]:         # all except current last
                if self._timeout():
                    return children
                others = [o for o in block if o != cand]

                blb = self._brucker_after_lb(cand, others, g)
                if blb >= self.ub:
                    continue

                if any(g.is_fixed(cand, o) for o in others):
                    continue

                child = g.copy()
                for o in others:
                    child.fix_arc(o, cand)       # every other → cand
                if not child.compute_heads_and_tails():
                    continue
                lb = max(blb, lower_bound(child))
                if lb < self.ub:
                    children.append(self._make_node(child, lb, depth+1))

        return children

    def _branch_pair(self, g: DisjunctiveGraph, depth: int) -> list[BBNode]:
        """Binary branching on most critical unfixed pair."""
        pair = g.most_critical_pair()
        if not pair: return []
        a, b = pair
        children: list[BBNode] = []
        for i, j in [(a,b),(b,a)]:
            c = g.copy(); c.fix_arc(i, j)
            if c.compute_heads_and_tails():
                lb = lower_bound(c)
                if lb < self.ub:
                    children.append(self._make_node(c, lb, depth+1))
        return children

    def _result(self) -> SolverResult:
        gap = 0.0
        if self.inst.bks and self.inst.bks > 0:
            gap = (self.ub - self.inst.bks)/self.inst.bks*100
        return SolverResult(instance_name=self.inst.name, makespan=self.ub,
            schedule=self.best, computation_time=self._elapsed(),
            nodes_explored=self.nodes, optimal_proven=self.optimal,
            bks=self.inst.bks, gap_vs_bks=gap)

In [ ]:
def run_large_instances():
    """Run B&B on large instances (LA16-LA20) with memory management."""
    results = []
    
    # Check for existing results (resume support)
    completed = set()
    if os.path.exists(OUTPUT_PATH):
        df_existing = pd.read_csv(OUTPUT_PATH)
        completed = set(df_existing['instance'].values)
        results = df_existing.to_dict('records')
        logger.info(f"Resuming: {len(completed)} instances already completed")
    
    start_time = time.time()
    
    for idx, (name, data) in enumerate(INSTANCES.items(), 1):
        if name in completed:
            logger.info(f"[{idx}/{len(INSTANCES)}] {name}: SKIPPED (already completed)")
            continue
        
        # Force garbage collection before each instance
        gc.collect()
        
        logger.info(f"\n{'='*60}")
        logger.info(f"[{idx}/{len(INSTANCES)}] Solving {name} (BKS={BKS[name]})")
        logger.info(f"⚠️ Large instance (10x10) - timeout expected")
        logger.info(f"{'='*60}")
        
        try:
            # Parse instance
            lines = data.strip().split('\n')
            hdr = lines[0].strip().split()
            nj, nm = int(hdr[0]), int(hdr[1])
            
            # Build jobs list
            jobs = []
            for j in range(nj):
                toks = lines[j+1].strip().split()
                job_ops = []
                for k in range(nm):
                    machine = int(toks[2*k])
                    ptime = int(toks[2*k+1])
                    job_ops.append((machine, ptime))
                jobs.append(job_ops)
            
            # Create instance
            instance = JSSPInstance(
                name=name,
                n_jobs=nj,
                n_machines=nm,
                jobs=jobs
            )
            
            # Solve with B&B (reduced timeout)
            solver = BranchAndBoundSolver(instance, timeout=TIMEOUT)
            result = solver.solve()
            
            # Store result
            makespan = result.best_makespan if result.best_makespan is not None else -1
            gap = ((makespan - BKS[name]) / BKS[name] * 100) if makespan > 0 and BKS[name] > 0 else -1
            
            result_dict = {
                'instance': name,
                'size': f"{nj}x{nm}",
                'bks': BKS[name],
                'makespan': makespan,
                'gap_pct': round(gap, 2) if gap >= 0 else -1,
                'optimal': result.optimal_proven,
                'time_s': result.computation_time_seconds,
                'nodes': result.nodes_explored
            }
            results.append(result_dict)
            
            # Save after each instance
            df = pd.DataFrame(results)
            df.to_csv(OUTPUT_PATH, index=False)
            
            # Log result
            status = "OPTIMAL" if result.optimal_proven else "TIMEOUT/FEASIBLE"
            logger.info(f"{status}: makespan={makespan}, BKS={BKS[name]}, gap={gap:.2f}%, time={result.computation_time_seconds:.1f}s, nodes={result.nodes_explored}")
            
            # Force cleanup
            del solver, result, instance
            gc.collect()
            
        except MemoryError as e:
            logger.error(f"OUT OF MEMORY on {name}: {e}")
            results.append({
                'instance': name,
                'size': f"{nj}x{nm}",
                'bks': BKS[name],
                'makespan': -1,
                'gap_pct': -1,
                'optimal': False,
                'time_s': -1,
                'nodes': -1
            })
            gc.collect()
            
        except Exception as e:
            logger.error(f"Error solving {name}: {e}")
            import traceback
            traceback.print_exc()
            results.append({
                'instance': name,
                'size': 'ERROR',
                'bks': BKS[name],
                'makespan': -1,
                'gap_pct': -1,
                'optimal': False,
                'time_s': -1,
                'nodes': -1
            })
    
    total_time = time.time() - start_time
    
    # Summary
    df = pd.DataFrame(results)
    optimal_count = df['optimal'].sum()
    feasible_count = (df['makespan'] > 0).sum()
    
    logger.info(f"\n{'='*60}")
    logger.info(f"LARGE INSTANCES COMPLETE")
    logger.info(f"{'='*60}")
    logger.info(f"Total instances: {len(INSTANCES)}")
    logger.info(f"Proven optimal: {optimal_count}")
    logger.info(f"Feasible solutions: {feasible_count}")
    logger.info(f"Total time: {total_time/3600:.2f} hours")
    logger.info(f"Results saved to: {OUTPUT_PATH}")
    
    return df

# Run the experiment
print("Starting large instances execution...")
print("⚠️ These instances are very challenging - expect timeouts!")
results_df = run_large_instances()
results_df